In [ ]:
# ----------- CREATING THE DIFFERENT CONTENT-BASED FILTERING MODELS ----------- #
# Assigning the different machine learning algorithms to be implemented in the models (incl. hyperparameters) to a dictionary
ml_algorithms = {"Linear regression": LinearRegression(), "Lasso": Lasso(alpha=1.0, max_iter=10000), 
                 "KNN_4": KNeighborsRegressor(n_neighbors=4),
                 "RFR": RandomForestRegressor(n_estimators=1000, n_jobs=3, max_features=3, random_state=0),
                 "SVR": SVR(C=1.0)}

# Saving lists that I later use to construct a dataframe containing the performances of the models
CBF_models_listed = []
RMSE_CBF_listed = []

# For every machine learning algorithm in the dictionary:
for name, ml_alg in ml_algorithms.items():
    # Create an empty list for predictions
    CBF_predictions = []

    # For each user in the training dataset:
    for i, x in enumerate(x_train_listed):
        # Fit a machine learning model
        ml_alg.fit(x_train_listed[i], y_train_listed[i])
        # Predict all the ratings for this user for all policys
        prediction = ml_alg.predict(all_policys)
        prediction = np.clip(prediction, 1, 5)  # Predictions must be minimum 1, maximum 5
        # Append all the predictions to the predictions list
        CBF_predictions.append(prediction)

    # Create a dataframe with the predictions
    df_predict = pd.DataFrame(CBF_predictions, index=user_ids, columns=policy_ids)

    # Create a dataframe with only the predictions for the policys-user combinations that appear in the validation set
    num_actual = validation_matrix.to_numpy().flatten()[validation_matrix.notna().to_numpy().flatten()]
    num_predict = df_predict.to_numpy().flatten()[validation_matrix.notna().to_numpy().flatten()]

    # Calculate the RMSE for the content-based filtering model and add the result to the lists
    RMSE_CBF_listed.append(sqrt(mean_squared_error(num_predict, num_actual)))
    CBF_models_listed.append(name)


# Printing the results
RMSE_CBF_df = pd.DataFrame({"Model": CBF_models_listed, "RMSE": RMSE_CBF_listed})
print("RMSE of different content-based filtering models without the year of release feature:")
RMSE_CBF_df



In [ ]:

# Running the best content-based filtering model so far
model = Lasso(alpha=1.0, max_iter=12000)
CBF_predictions = []

# For each user in the training dataset:
for i, j in enumerate(x_train_listed):
    model.fit(x_train_listed[i], y_train_listed[i])
    prediction = model.predict(all_policys)
    prediction = np.clip(prediction, 1, 5)
    CBF_predictions.append(prediction)

# Creating a dataframe for the predictions
CBF_model = pd.DataFrame(CBF_predictions, index=user_ids, columns=policy_ids)

In [ ]:

# MODELLING: Predicting ratings for every user with K Nearest Neighbours
# Models with a different number of neighbors
ml_algorithms = {'kNN-5': 5, 'kNN-10': 10, 'kNN-20': 20, 'kNN-30': 30, 'kNN-40': 40, "kNN-60": 60}

models_CF = []
RMSE_CF = []

# Training the models and predicting for the users and polcy in the validation data
for name, num_neighbours in ml_algorithms.items():
    predictions = []
    users_rated_policy =[]
    users_sorted =[]
    # For every rating in the validation datas
    for index, row in X_val.iterrows():
        if row["Policy"] in X_train["Policy"].unique():
            users_rated_policy.append(X_train.loc[X_train['Policy'] == row['Policy'], 'BrukerID']) 
            for x in users_rated_policy:
                users_sorted.append(user_dist_matrix.iloc[row['BrukerID'],x-1])
            
            users_sorted.sort_values() 
            # Select the nearest neighbours
            nearest_neighbours = users_sorted[:num_neighbours]
            # Extract the nearest neighbours' ratings data
            nn_data = train_df.loc[train_df['BrukerID'].isin(nearest_neighbours.index.to_list())]
            # Calculate the weighted average of the nearest neighbours' ratings
            nearest_neighbours_avg_rating = np.average(nn_data.loc[train_df[' Policy'] == row['Policy'], 'Rangerings'],
                                                       axis=0, weights=(1/nearest_neighbours.values))
        else:
            # There is a small chance that a few policys in the validation set might not appear in the training set.
            # I therefore predict that the user will rate these policys with the average rating for all policys
            nearest_neighbours_avg_rating = 4   # Must be changed!

        # Appending the prediction to the list of predictions
        if not np.isnan(nearest_neighbours_avg_rating):
            predictions.append(nearest_neighbours_avg_rating)
        else:
            predictions.append(3)

    models_CF.append(name)
    RMSE_CF.append(sqrt(mean_squared_error(y_val, predictions)))


# Displaying the results
RMSE_CF_dict = {"Model": models_CF, "RMSE": RMSE_CF}
RMSE_CF_df = pd.DataFrame(RMSE_CF_dict)
RMSE_CF_df

In [ ]:
 K-value of 40, the model reached a RMSE of 0.9588.

In [ ]:

# Visualizing how the number of neighbors effect the root mean sqaured error
fig7, ax7 = plt.subplots()
ax7.plot(RMSE_CF_df.Model, RMSE_CF_df.RMSE, label="RMSE", color='darkred', linewidth=2)
plt.xlabel("Number of nearest neighbors", labelpad=18)
plt.ylabel("Root mean squared error", labelpad=15)
plt.title("K-value effect on RMSE for collaborative filtering models")
fig7.set_figheight(10)
fig7.set_figwidth(16)
plt.show()


In [ ]:
# Rerunning the best model so far (kNN-40) and storing the prediction results
best_CF_model = []
RMSE_best_CF = []

# Training the models and predicting for the users and policys in the validation data
CF_predictions = []

# For every  in the validation data
for index, row in X_val.iterrows():
    # If that policy is in the training data
    if row["Policy"] in X_train["Policy"].unique():
        users_rated_policy = X_train.loc[X_train['Policy'] == row['Policy'], 'BrukerID']
        users_sorted = (user_dist_matrix.loc[row['BrukerID'], users_rated_policy].sort_values())
        nearest_neighbours = users_sorted[:40]
        # Extract the nearest neighbours' ratings data
        nn_data = train_df.loc[train_df['BrukerID'].isin(nearest_neighbours.index.to_list())]
        # Calculate the weighted average of the nearest neighbours' ratings
        nearest_neighbours_avg_rating = np.average(nn_data.loc[train_df['Policy'] == row['Policy'], 'Rangering'],
                                                   axis=0, weights=(1/nearest_neighbours))
    else:
        nearest_neighbours_avg_rating = 4   # Must be changed!

    # Appending the prediction to the list of predictions
    if not np.isnan(nearest_neighbours_avg_rating):
        CF_predictions.append(nearest_neighbours_avg_rating)
    else:
        CF_predictions.append(4)

In [ ]:

# Building the hybrid recommender: Collaborative Filtering
CF_predictions_test = []
for index, row in X_test.iterrows():
    if row["Policy"] in X_train["Policy"].unique():
        users_rated_policy = X_train.loc[X_train['Policy'] == row['Policy'], 'BrukerID']
        users_sorted = (user_dist_matrix.loc[row['BrukerID'], users_rated_policy].sort_values())
        n_neighbours = users_sorted[:40]
        nn_data = train_df.loc[train_df['BrukerID'].isin(n_neighbours.index.to_list())]
        nearest_neighbours_avg_rating = np.average(nn_data.loc[train_df['Policy'] == row['Policy'], 'Rangering'],
                                                   axis=0, weights=(1/n_neighbours))
    else:
        nearest_neighbours_avg_rating = train_df["Rangering"].mean()

    # appending the prediction to the list
    if not np.isnan(nearest_neighbours_avg_rating):
        CF_predictions_test.append(nearest_neighbours_avg_rating)
    else:
        CF_predictions_test.append(4)

print("RMSE KNN_40:", sqrt(mean_squared_error(y_test, CF_predictions_test)))

In [ ]:

# Building the hybrid recommender: Content-Based filtering
# Extracting the predictions for the policys and users in the test data
# from the CBF dataframe (which contains predictions for all policys and all users)
CBF_predictions_test = []
for index, row in X_test.iterrows():
    user_predictions = CBF_improved_model.loc[row["BrukerID"], row["Policy"]]
    CBF_predictions_test.append(user_predictions)

print("RMSE Lasso:", sqrt(mean_squared_error(y_test, CBF_predictions_test)))

In [ ]:
# Calculating the hybrid recommendations
hybrid_predictions_test = (np.array([y_pred * 0.35 for y_pred in np.array(CBF_predictions_test)]) 
                           + np.array([y_pred * 0.65 for y_pred in np.array(CF_predictions_test)]))

# Displaying the test results from training the Hybrid Recommender on test data
print(f"RMSE hybrid recommendations (test data): {sqrt(mean_squared_error(y_test, hybrid_predictions_test))} ")